In [ ]:
"""
=============================================================================
PIPELINE FINAL + GENERATION DES FIGURES ET TABLES — jcp-4466772
=============================================================================
Configuration figee : deux ecosystemes, pre-filtre vman, k = 18.
Justifiee par la comparaison de quatre executions : meilleur lift AU-PR
(0.278), meilleur MCC (0.832), meilleur rappel operationnel (0.896), et
seule configuration ou la selection depasse l'intervalle de confiance.

PRODUIT
  Tables 1 a 11 : en CSV et en Markdown pret a coller dans Word
  Figures 2 a 6 : en PNG 600 dpi, PDF et SVG
  final_results.json : toute valeur du manuscrit y est tracable
  MANUSCRIPT_VALUES.md : correspondance table/figure -> section du papier

CORRECTION : le message de synthese de la version precedente affirmait en dur
que la selection ne depasse pas le bruit. Il est desormais calcule.
=============================================================================
"""
import os, re, json, itertools, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             confusion_matrix, brier_score_loss)
from scipy.stats import binomtest, spearmanr, beta as _beta
from catboost import CatBoostClassifier
import shap

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CFG = dict(
    k_features=18, proxy_scope="vman",
    k_sweep=[5, 10, 15, 18, 25, 40],
    test_frac=0.25, val_frac=0.20, name_sim=0.85,
    n_boot=1000, n_boot_paired=2000,
    prevalences=[0.01, 0.001, 0.0001], target_prevalence=0.001,
    min_recall_val=0.70, max_transfer_gap=0.20,
    n_random_draws=15, n_stability_runs=10, n_robustness_runs=10,
)

try:
    from google.colab import drive; drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/JCP_revision"
except Exception:
    OUT = "./JCP_revision"
FIG = f"{OUT}/figures"; TAB = f"{OUT}/tables"
for d in (OUT, FIG, TAB): os.makedirs(d, exist_ok=True)
CACHE = f"{OUT}/dataset_v5.csv"
R = {"config": CFG}
MD = {}                      # tables en markdown, pour le manuscrit

BLUE, ORANGE, GREY = "#1F4E9C", "#E8973A", "#8C8C8C"

def to_md(dfr, name, caption, floatfmt=4):
    """Exporte une table en CSV et en Markdown pret a coller."""
    d = dfr.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]): d[c] = d[c].round(floatfmt)
    d.to_csv(f"{TAB}/{name}.csv", index=False)
    md = f"**{caption}**\n\n| " + " | ".join(str(c) for c in d.columns) + " |\n"
    md += "|" + "|".join(["---"] * len(d.columns)) + "|\n"
    for _, r_ in d.iterrows():
        md += "| " + " | ".join("" if pd.isna(v) else str(v) for v in r_) + " |\n"
    MD[name] = md
    return d

# =============================================================================
# OUTILS
# =============================================================================
META = ["package", "version", "ecosystem", "t_p", "maintainer", "label",
        "group_id", "canon", "advisory_id"]
LEAK = ["exists", "error_type", "downloads", "last_serial", "yanked",
        "deprecated", "status", "is_removed", "_npmUser"]

def numeric_pit(part):
    X = part.drop(columns=[c for c in META if c in part.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number])
    return X[[c for c in X.columns if c.startswith(("vman_", "pit_"))
              and not any(re.search(p, c) for p in LEAK)]]

def metrics(y, s, thr=0.5):
    yp = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(y)
    p = precision_score(y, yp, pos_label=1, zero_division=0)
    r_ = recall_score(y, yp, pos_label=1, zero_division=0)
    f1 = f1_score(y, yp, pos_label=1, zero_division=0)
    if p + r_ > 0: assert abs(2*p*r_/(p+r_) - f1) < 1e-9
    return dict(n=len(y), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                accuracy=accuracy_score(y, yp), precision=p, recall=r_, f1=f1,
                mcc=matthews_corrcoef(y, yp), auc_roc=roc_auc_score(y, s),
                au_pr=average_precision_score(y, s), au_pr_baseline=float(np.mean(y)),
                au_pr_lift=average_precision_score(y, s) - float(np.mean(y)),
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                fpr=fp/(tn+fp) if tn+fp else np.nan,
                brier=brier_score_loss(y, s), threshold=float(thr))

def boot_ci(y, s, fn, n=CFG["n_boot"]):
    rng = np.random.default_rng(SEED); y, s = np.asarray(y), np.asarray(s); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1: v.append(fn(y[i], s[i]))
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

def fpr_upper(fp, n_ben, conf=0.95):
    if n_ben <= 0: return np.nan
    if fp <= 0: return 3.0 / n_ben
    if fp >= n_ben: return 1.0
    return float(_beta.ppf(conf, fp + 1, n_ben - fp))

def operational(y, s, thr, pi):
    tn, fp, fn, tp = confusion_matrix(y, (s >= thr).astype(int), labels=[0,1]).ravel()
    tpr = tp/(tp+fn) if tp+fn else np.nan
    out = {}
    for lab, f_ in [("point", fp/(fp+tn) if fp+tn else np.nan),
                    ("upper95", fpr_upper(fp, fp+tn))]:
        den = tpr*pi + f_*(1-pi)
        out[lab] = {"TPR": tpr, "FPR": f_,
                    "PPV": (tpr*pi)/den if den and den > 0 else np.nan,
                    "alerts_per_10k": 10000*den if den == den else np.nan,
                    "true_positives_per_10k": 10000*tpr*pi,
                    "missed_per_10k": 10000*pi*(1-tpr)}
    return out

def mcnemar(y, pa, pb):
    a_ok, b_ok = (pa == y), (pb == y)
    b = int(np.sum(a_ok & ~b_ok)); c = int(np.sum(~a_ok & b_ok))
    return {"b": b, "c": c,
            "p_exact": float(binomtest(min(b, c), b + c, .5).pvalue) if b + c else 1.0}

def paired_bootstrap(y, sa, sb, n=CFG["n_boot_paired"]):
    rng = np.random.default_rng(SEED); y = np.asarray(y); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        d.append(average_precision_score(y[i], sa[i]) -
                 average_precision_score(y[i], sb[i]))
    d = np.array(d)
    return {"mean_diff": float(d.mean()),
            "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            "p_two_sided": float(2 * min((d <= 0).mean(), (d >= 0).mean()))}

jaccard = lambda a, b: len(set(a) & set(b)) / len(set(a) | set(b)) if (a or b) else np.nan

# categories et definitions, pour la Table 4
CATEGORY_RULES = [
    (r"author|maintainer|email",                    "Authorship & maintainers"),
    (r"description|summary|keyword",                "Description & keywords"),
    (r"days_since|gap_days|releases_\d+d|prior_rel", "Release history"),
    (r"ver_|version",                               "Versioning"),
    (r"depend",                                     "Dependencies"),
    (r"script|install_hook|engines",                "Build & install"),
]
def category_of(f):
    for pat, cat in CATEGORY_RULES:
        if re.search(pat, f): return cat
    return "Identity & links"

DEFINITIONS = {
    "pit_long_description_len": "Character length of the long description at t_p",
    "pit_n_maintainers": "Number of maintainer accounts listed at t_p",
    "pit_has_repository": "Repository field present at t_p (boolean)",
    "pit_days_since_first": "Days between the first release and t_p",
    "pit_has_install_hook": "Install, preinstall or postinstall hook declared (boolean)",
    "pit_ver_is_zero_major": "Version string begins with 0. (boolean)",
    "pit_n_keywords": "Number of declared keywords at t_p",
    "pit_has_engines": "Engines field present in the manifest (boolean)",
    "pit_std_gap_days": "Standard deviation of inter-release intervals before t_p",
    "pit_description_len": "Character length of the summary at t_p",
    "pit_median_gap_days": "Median inter-release interval before t_p",
    "pit_author_email_domain_score": ("Ordinal score of the author email domain "
        "(organisational, free-mail, disposable, missing); local part discarded"),
    "vman_description_len": "Character length of the description in the published manifest",
    "pit_name_has_scope": "Package name carries an organisation scope (boolean)",
    "pit_scripts_max_len": "Longest declared script, in characters",
    "pit_n_dependencies": "Declared runtime dependencies at t_p",
    "pit_description_entropy": "Shannon entropy of the summary at t_p",
    "vman_author_len": "Character length of the author field in the published manifest",
    "pit_name_entropy": "Shannon entropy of the canonicalised package name",
    "pit_n_dev_dependencies": "Declared development dependencies at t_p",
    "pit_releases_30d": "Releases published in the 30 days before t_p",
}
definition_of = lambda f: DEFINITIONS.get(f, "See Supplementary Table S3")

# =============================================================================
# 1. DONNEES, GROUPES, SPLIT
# =============================================================================
df = pd.read_csv(CACHE, low_memory=False)
df["t_p"] = pd.to_datetime(df["t_p"], utc=True, errors="coerce")
df = df.dropna(subset=["t_p"]).fillna(0).reset_index(drop=True)
raw_comp = df.groupby(["ecosystem", "label"]).size().to_dict()

parts, WIN = [], {}
for eco in df.ecosystem.unique():
    sub = df[df.ecosystem == eco]; mal = sub[sub.label == 1]
    if len(mal) < 50:
        parts.append(sub); WIN[eco] = None; continue
    lo, hi = mal.t_p.quantile(.05), mal.t_p.quantile(.95)
    parts.append(sub[(sub.t_p >= lo) & (sub.t_p <= hi)])
    WIN[eco] = [str(lo.date()), str(hi.date())]
df = pd.concat(parts, ignore_index=True)
print(f"{len(df)} versions apres appariement temporel | {WIN}")

canon = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
def nsim(a, b):
    if a == b: return 1.0
    la, lb = len(a), len(b)
    if not la or not lb: return 0.0
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return 1.0 - prev[lb]/max(la, lb)
class UF:
    def __init__(s): s.p = {}
    def find(s, x):
        s.p.setdefault(x, x)
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[ra] = rb

uf = UF(); df["canon"] = df["package"].map(canon)
for _, r_ in df.iterrows():
    uf.find(("pkg", r_["canon"]))
    m = str(r_.get("maintainer") or "").strip().lower()
    if m and m not in ("nan", "0", ""): uf.union(("pkg", r_["canon"]), ("mnt", m))
blocks = {}
for n in sorted(df["canon"].unique()): blocks.setdefault((n[:2], len(n)//4), []).append(n)
npairs = 0
for blk in blocks.values():
    for a, b in itertools.combinations(blk, 2):
        if abs(len(a)-len(b)) <= 3 and nsim(a, b) >= CFG["name_sim"]:
            uf.union(("pkg", a), ("pkg", b)); npairs += 1
df["group_id"] = df["canon"].map(lambda n: str(uf.find(("pkg", n))))
sz = df.group_id.value_counts()

i_tv, i_te = next(GroupShuffleSplit(1, test_size=CFG["test_frac"], random_state=SEED)
                  .split(df, df.label, groups=df.group_id))
trval, test = df.iloc[i_tv].copy(), df.iloc[i_te].copy()
i_tr, i_va = next(GroupShuffleSplit(1, test_size=CFG["val_frac"], random_state=SEED)
                  .split(trval, trval.label, groups=trval.group_id))
train, val = trval.iloc[i_tr].copy(), trval.iloc[i_va].copy()
for a, b, nm in [(train, val, "tr/va"), (train, test, "tr/te"), (val, test, "va/te")]:
    assert not (set(a.group_id) & set(b.group_id)), f"fuite de groupe {nm}"
y_tr, y_va, y_te = train.label.values, val.label.values, test.label.values
print(f"{df.group_id.nunique()} groupes | train {len(train)} val {len(val)} "
      f"test {len(test)} ({int(y_te.sum())} mal / {int((1-y_te).sum())} ben)")

# ---- TABLE 1 : composition et partitions --------------------------------
t1 = []
for eco in sorted(df.ecosystem.unique()):
    sub = df[df.ecosystem == eco]
    t1.append({"Ecosystem": eco.upper(), "Versions": len(sub),
               "Malicious": int(sub.label.sum()), "Benign": int((1-sub.label).sum()),
               "Malicious %": f"{sub.label.mean():.1%}"})
t1.append({"Ecosystem": "Total", "Versions": len(df), "Malicious": int(df.label.sum()),
           "Benign": int((1-df.label).sum()), "Malicious %": f"{df.label.mean():.1%}"})
T1 = to_md(pd.DataFrame(t1), "table01_dataset",
           "Table 1. Dataset composition after temporal matching.")
t1b = [{"Partition": n, "Versions": len(p), "Malicious": int(p.label.sum()),
        "Benign": int((1-p.label).sum())}
       for n, p in [("Training", train), ("Validation", val), ("Test", test)]]
T1B = to_md(pd.DataFrame(t1b), "table01b_partitions",
            "Table 1b. Partition sizes. Partitioning operates on group identifiers, "
            "so sizes are not exact fractions of the dataset.")
print(T1.to_string(index=False))
R["dataset"] = {"n_versions": len(df), "n_packages": int(df.package.nunique()),
                "n_groups": int(df.group_id.nunique()), "largest_group": int(sz.iloc[0]),
                "largest_group_pct": float(sz.iloc[0]/len(df)),
                "similar_name_pairs": npairs, "windows": WIN,
                "n_train": len(train), "n_val": len(val), "n_test": len(test),
                "test_malicious": int(y_te.sum()), "test_benign": int((1-y_te).sum()),
                "before_temporal_matching": {str(k): int(v) for k, v in raw_comp.items()}}

# =============================================================================
# 2. PRE-FILTRE ET SELECTION
# =============================================================================
F_FULL = list(numeric_pit(train).columns)
Xn = numeric_pit(train)
cols = [c for c in Xn.columns if c.startswith("vman_")] \
       if CFG["proxy_scope"] == "vman" else list(Xn.columns)
PROXIES = [c for c in cols
           if (train.groupby("ecosystem")[c].std().fillna(0) == 0).any()] \
          if df.ecosystem.nunique() > 1 else []
F_POST = [c for c in F_FULL if c not in PROXIES]
print(f"\nespace complet {len(F_FULL)} | proxies {len(PROXIES)} | post-filtre {len(F_POST)}")

def make_prep(feats):
    X0 = numeric_pit(train)[feats]
    imp = SimpleImputer(strategy="median").fit(X0)
    A = imp.transform(X0)
    LOq, HIq = np.nanpercentile(A, 1, 0), np.nanpercentile(A, 99, 0)
    scl = StandardScaler().fit(np.clip(A, LOq, HIq))
    return lambda part: pd.DataFrame(
        scl.transform(np.clip(imp.transform(
            numeric_pit(part).reindex(columns=feats, fill_value=0)), LOq, HIq)),
        columns=feats, index=part.index)

prep_post, prep_full = make_prep(F_POST), make_prep(F_FULL)
Xtr, Xva, Xte = prep_post(train), prep_post(val), prep_post(test)
XtrF, XteF = prep_full(train), prep_full(test)
HP0 = dict(iterations=400, learning_rate=0.1, depth=6, verbose=0,
           random_state=SEED, auto_class_weights="Balanced")
def score_on(feats, Xf, Xe, hp=None):
    return CatBoostClassifier(**(hp or HP0)).fit(Xf[feats], y_tr
                                                 ).predict_proba(Xe[feats])[:, 1]

sel_m = CatBoostClassifier(**HP0).fit(Xtr, y_tr)
RANK = pd.Series(sel_m.get_feature_importance(), index=F_POST).sort_values(ascending=False)
K = min(CFG["k_features"], len(F_POST))
SEL = list(RANK.head(K).index)

# ---- TABLE 2 : reduction de l'espace ------------------------------------
T2 = to_md(pd.DataFrame([
    {"Stage": "Extracted point-in-time attributes", "Features in": "-",
     "Features out": len(F_FULL), "Removed": "-"},
    {"Stage": f"Ecosystem pre-filter ({CFG['proxy_scope']} scope)",
     "Features in": len(F_FULL), "Features out": len(F_POST), "Removed": len(PROXIES)},
    {"Stage": "Embedded selection, top-k (a priori)",
     "Features in": len(F_POST), "Features out": K, "Removed": len(F_POST)-K},
]), "table02_reduction",
    f"Table 2. Feature space reduction: {len(F_FULL)} to {K} attributes "
    f"({100*(1-K/len(F_FULL)):.1f}% reduction).")
print(T2.to_string(index=False))

Xtr_s, Xva_s, Xte_s = Xtr[SEL], Xva[SEL], Xte[SEL]

# =============================================================================
# 3. MODELE
# =============================================================================
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = {"depth": [4, 6, 8], "learning_rate": [0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5], "iterations": [500]}
gs = GridSearchCV(CatBoostClassifier(verbose=0, random_state=SEED,
                                     auto_class_weights="Balanced"),
                  grid, scoring="average_precision", cv=cv, n_jobs=-1)
gs.fit(Xtr_s, y_tr, groups=train.group_id.values)
HP = {**gs.best_params_, "verbose": 0, "random_state": SEED,
      "auto_class_weights": "Balanced"}
model = CatBoostClassifier(**HP).fit(Xtr_s, y_tr)
s_te = model.predict_proba(Xte_s)[:, 1]
s_va = model.predict_proba(Xva_s)[:, 1]
M = metrics(y_te, s_te)
CI = {"auc_roc": boot_ci(y_te, s_te, roc_auc_score),
      "au_pr": boot_ci(y_te, s_te, average_precision_score)}
CI_W = CI["au_pr"][1] - CI["au_pr"][0]
print(f"\nAU-PR {M['au_pr']:.4f} CI [{CI['au_pr'][0]:.4f},{CI['au_pr'][1]:.4f}] "
      f"| AUC {M['auc_roc']:.4f} | MCC {M['mcc']:.4f}")
R["hyperparameters"] = {"grid": grid, "selected": gs.best_params_,
                        "cv_score_au_pr": float(gs.best_score_)}

# ---- SHAP (necessaire pour la Table 3) ----------------------------------
sv = shap.TreeExplainer(model).shap_values(Xte_s)
if isinstance(sv, list): sv = sv[1]
SHAPV = pd.Series(np.abs(sv).mean(0), index=SEL).sort_values(ascending=False)

# ---- TABLE 3 : les k features -------------------------------------------
T3 = to_md(pd.DataFrame([
    {"#": i, "Feature": f, "Category": category_of(f),
     "Definition": definition_of(f), "Mean |SHAP|": SHAPV[f]}
    for i, f in enumerate(SHAPV.index, 1)]), "table03_features",
    f"Table 3. The {K} selected features, ordered by mean absolute SHAP value "
    f"(CatBoost, test partition, n = {M['n']}).")
print(f"\nTable 3 : {K} features, dominante = {SHAPV.index[0]} ({SHAPV.iloc[0]:.3f})")

# ---- TABLE 4 : performance ----------------------------------------------
T4 = to_md(pd.DataFrame([{
    "Metric": k, "Value": v} for k, v in [
    ("AU-PR", M["au_pr"]), ("AU-PR 95% CI", f"[{CI['au_pr'][0]:.4f}, {CI['au_pr'][1]:.4f}]"),
    ("AU-PR baseline", M["au_pr_baseline"]), ("AU-PR lift", M["au_pr_lift"]),
    ("AUC-ROC", M["auc_roc"]),
    ("AUC-ROC 95% CI", f"[{CI['auc_roc'][0]:.4f}, {CI['auc_roc'][1]:.4f}]"),
    ("MCC", M["mcc"]), ("F1 (positive class)", M["f1"]),
    ("Accuracy", M["accuracy"]), ("Precision", M["precision"]),
    ("Recall", M["recall"]), ("Specificity", M["specificity"]),
    ("False-positive rate", M["fpr"]), ("Brier score", M["brier"]),
    ("Confusion matrix (TN/FP/FN/TP)",
     f"{M['TN']} / {M['FP']} / {M['FN']} / {M['TP']}")]]),
    "table04_performance",
    f"Table 4. Model performance on the test partition (n = {M['n']}), "
    f"CatBoost with {K} features under the corrected protocol.")
R["performance"] = {**M, "ci95": CI, "ci_width_au_pr": float(CI_W)}

# =============================================================================
# 4. SEUIL ET PREVALENCE
# =============================================================================
pi = CFG["target_prevalence"]
cands = []
for t in np.unique(np.round(s_va, 3)):
    mv = metrics(y_va, s_va, t)
    if mv["recall"] < CFG["min_recall_val"]: continue
    ov = operational(y_va, s_va, t, pi)["upper95"]
    if np.isfinite(ov["PPV"]):
        cands.append({"threshold": float(t), "ppv_val": ov["PPV"]})
thr = float(pd.DataFrame(cands).sort_values("ppv_val").iloc[-1]["threshold"]) \
      if cands else 0.5
rule = f"max PPV at {pi} on validation, recall >= {CFG['min_recall_val']}"
mv, mt = metrics(y_va, s_va, thr), metrics(y_te, s_te, thr)
gap = mv["recall"] - mt["recall"]
if abs(gap) > CFG["max_transfer_gap"]:
    print(f"\n!! ecart de transfert {gap:+.3f} : seuil ramene a 0.5")
    thr, rule = 0.5, "default 0.5 (validation-optimised threshold did not transfer)"
    mv, mt = metrics(y_va, s_va, thr), metrics(y_te, s_te, thr)
    gap = mv["recall"] - mt["recall"]

rows = []
for p_ in CFG["prevalences"]:
    o = operational(y_te, s_te, thr, p_)
    for lab in ("point", "upper95"):
        rows.append({"Prevalence": f"{p_:.2%}".rstrip("0").rstrip("."),
                     "FPR estimate": lab, "Recall": o[lab]["TPR"],
                     "FPR": o[lab]["FPR"], "PPV": o[lab]["PPV"],
                     "Alerts per 10,000": o[lab]["alerts_per_10k"],
                     "True positives per 10,000": o[lab]["true_positives_per_10k"],
                     "Missed per 10,000": o[lab]["missed_per_10k"]})
PT = pd.DataFrame(rows)
op = operational(y_te, s_te, thr, pi)["upper95"]
T5 = to_md(PT, "table05_prevalence",
    f"Table 5. Alert quality at operational prevalence, threshold {thr:.3f} "
    f"({rule}). A false-positive rate measured at zero on {mt['FP']+mt['TN']} benign "
    f"instances is not zero; the upper95 rows apply the rule of three.")
print(f"\nSeuil {thr:.3f} | rappel val {mv['recall']:.3f} -> test {mt['recall']:.3f} "
      f"(ecart {gap:+.3f}) | VPP a {pi:.1%} : {op['PPV']:.4f}")
R["operational"] = {"threshold": thr, "rule": rule, "transfer_gap": float(gap),
                    "metrics_val": mv, "metrics_test": mt,
                    "at_target_prevalence": op, "prevalence_table": PT.to_dict("records")}

# =============================================================================
# 5. DECOMPOSITION DU GAIN ET BALAYAGE DE k
# =============================================================================
rng = np.random.default_rng(SEED)
vf = []
for _ in range(CFG["n_random_draws"]):
    fs = list(rng.choice(F_FULL, size=min(K, len(F_FULL)), replace=False))
    s = score_on(fs, XtrF, XteF, hp=HP)
    vf.append((average_precision_score(y_te, s), roc_auc_score(y_te, s),
               sum(1 for f in fs if f in PROXIES)))
vf = np.array(vf)
vp = [average_precision_score(y_te, score_on(
        list(rng.choice(F_POST, size=min(K, len(F_POST)), replace=False)),
        Xtr, Xte, hp=HP)) for _ in range(CFG["n_random_draws"])]
g_full = M["au_pr"] - float(np.median(vf[:, 0]))
g_post = M["au_pr"] - float(np.median(vp))
sel_sig = g_post >= CI_W

T6 = to_md(pd.DataFrame([
    {"Feature set": f"Random {K} of {len(F_FULL)} (full space)",
     "AU-PR": float(np.median(vf[:, 0])), "AUC-ROC": float(np.median(vf[:, 1])),
     "Mean proxies per draw": float(vf[:, 2].mean())},
    {"Feature set": f"Random {K} of {len(F_POST)} (post-filter)",
     "AU-PR": float(np.median(vp)), "AUC-ROC": np.nan, "Mean proxies per draw": 0.0},
    {"Feature set": f"Selected {K} (ours)", "AU-PR": M["au_pr"],
     "AUC-ROC": M["auc_roc"], "Mean proxies per draw": 0.0}]),
    "table06_gain_decomposition",
    f"Table 6. Gain decomposition, {CFG['n_random_draws']} random draws each. "
    f"The pre-filter contributes {g_full-g_post:+.4f} in AU-PR and the selection "
    f"{g_post:+.4f}, against a 95% bootstrap interval width of {CI_W:.4f}.")
print(f"\n[a] pre-filtre {g_full-g_post:+.4f} | selection {g_post:+.4f} | IC {CI_W:.4f}")
print(f"    -> la selection est {'au-dessus du' if sel_sig else 'sous le'} bruit")
R["gain_decomposition"] = {
    "random_full": {"n": len(F_FULL), "au_pr": float(np.median(vf[:,0])),
                    "auc": float(np.median(vf[:,1])),
                    "mean_proxies": float(vf[:,2].mean())},
    "random_post": {"n": len(F_POST), "au_pr": float(np.median(vp))},
    "prefilter_gain": float(g_full-g_post), "selection_gain": float(g_post),
    "selection_exceeds_ci": bool(sel_sig)}

sweep = []
for k in [k for k in CFG["k_sweep"] if k <= len(F_POST)]:
    s = score_on(list(RANK.head(k).index), Xtr, Xte, hp=HP)
    sweep.append({"k": k, "AU-PR": average_precision_score(y_te, s),
                  "AUC-ROC": roc_auc_score(y_te, s)})
SWEEP = pd.DataFrame(sweep)
ref = float(SWEEP.loc[SWEEP.k == K, "AU-PR"].iloc[0])
SWEEP["Delta vs k=18"] = SWEEP["AU-PR"] - ref
SWEEP["Within CI"] = SWEEP["Delta vs k=18"].abs() <= CI_W
T7 = to_md(SWEEP, "table07_k_sweep",
    f"Table 7. Performance as a function of feature-set size. Differences from "
    f"k = {K} are compared with the bootstrap interval width ({CI_W:.4f}); k was "
    f"declared a priori and this sweep is reported, not used for the choice.")
print(SWEEP.round(4).to_string(index=False))
R["k_sweep"] = SWEEP.to_dict("records")

# =============================================================================
# 6. TESTS APPARIES, STABILITE, ABLATION PAR FAMILLE
# =============================================================================
ALT = {}
ALT["Mutual information"] = list(pd.Series(
    mutual_info_classif(Xtr, y_tr, random_state=SEED), index=F_POST
    ).sort_values(ascending=False).head(K).index)
_rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             random_state=SEED, n_jobs=-1).fit(Xtr, y_tr)
ALT["RF importance"] = list(pd.Series(_rf.feature_importances_, index=F_POST
                            ).sort_values(ascending=False).head(K).index)
_l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                         class_weight="balanced", max_iter=2000).fit(Xtr, y_tr)
ALT["L1 logistic"] = list(pd.Series(np.abs(_l1.coef_[0]), index=F_POST
                          ).sort_values(ascending=False).head(K).index)
rows, pred_ref = [], (s_te >= 0.5).astype(int)
for nm, fs in ALT.items():
    s_alt = score_on(fs, Xtr, Xte, hp=HP)
    pb = paired_bootstrap(y_te, s_te, s_alt); mc = mcnemar(y_te, pred_ref,
                                                           (s_alt >= 0.5).astype(int))
    rows.append({"Comparison": f"Ours vs {nm}",
                 "AU-PR (alt.)": average_precision_score(y_te, s_alt),
                 "Delta AU-PR": pb["mean_diff"], "CI low": pb["ci95"][0],
                 "CI high": pb["ci95"][1], "p (bootstrap)": pb["p_two_sided"],
                 "b": mc["b"], "c": mc["c"], "p (McNemar)": mc["p_exact"],
                 "Jaccard": jaccard(fs, SEL)})
PAIRED = pd.DataFrame(rows); nc = max(len(PAIRED), 1)
PAIRED["p (bootstrap, Bonf.)"] = (PAIRED["p (bootstrap)"]*nc).clip(upper=1.0)
PAIRED["p (McNemar, Bonf.)"] = (PAIRED["p (McNemar)"]*nc).clip(upper=1.0)
T8 = to_md(PAIRED[["Comparison", "AU-PR (alt.)", "Delta AU-PR", "CI low", "CI high",
                   "p (bootstrap, Bonf.)", "b", "c", "p (McNemar, Bonf.)", "Jaccard"]],
    "table08_paired_tests",
    f"Table 8. Paired comparison against alternative selectors at a matched budget "
    f"of {K} features. Bonferroni correction across {nc} comparisons. Positive "
    f"deltas favour our selection.")
print("\n[d] tests apparies :"); print(PAIRED.round(4).to_string(index=False))
R["paired_tests"] = {"n_comparisons": nc, "table": PAIRED.to_dict("records")}

uq = np.unique(train.group_id.values); sets = []
for _ in range(CFG["n_stability_runs"]):
    keep = set(rng.choice(uq, size=len(uq), replace=True))
    mk = np.array([g in keep for g in train.group_id.values])
    if len(np.unique(y_tr[mk])) < 2: continue
    m_ = CatBoostClassifier(**HP0).fit(Xtr[mk], y_tr[mk])
    sets.append(list(pd.Series(m_.get_feature_importance(), index=F_POST)
                     .sort_values(ascending=False).head(K).index))
jac = [jaccard(a, b) for a, b in itertools.combinations(sets, 2)]
freq = pd.Series([f for s in sets for f in s]).value_counts()/max(len(sets), 1)
print(f"[c] stabilite Jaccard {np.mean(jac):.3f} | {int((freq>=.8).sum())}/{K} stables")
R["stability"] = {"mean_jaccard": float(np.mean(jac)) if jac else None,
                  "n_stable_80pct": int((freq >= .8).sum()), "n_runs": len(sets)}

FAM = {}
for f in SEL: FAM.setdefault(category_of(f), []).append(f)
rows = [{"Family withheld": "None", "Features removed": 0, "AU-PR": M["au_pr"],
         "AUC-ROC": M["auc_roc"], "Delta AU-PR": 0.0}]
for nm, fs in FAM.items():
    rest = [f for f in SEL if f not in fs]
    if not rest: continue
    s = score_on(rest, Xtr, Xte, hp=HP)
    a = average_precision_score(y_te, s)
    rows.append({"Family withheld": nm, "Features removed": len(fs), "AU-PR": a,
                 "AUC-ROC": roc_auc_score(y_te, s), "Delta AU-PR": a - M["au_pr"]})
FAMT = pd.DataFrame(rows).sort_values("Delta AU-PR")
T9 = to_md(FAMT, "table09_family_ablation",
    f"Table 9. Metadata-family ablation. Deltas are compared with the bootstrap "
    f"interval width ({CI_W:.4f}).")
print(FAMT.round(4).to_string(index=False))
R["family_ablation"] = {"membership": FAM, "table": FAMT.to_dict("records")}

# =============================================================================
# 7. ROBUSTESSE ET GENERALISATION
# =============================================================================
def rob_label_noise(rate=0.10, n=CFG["n_robustness_runs"]):
    r2 = np.random.default_rng(SEED); v = []
    for _ in range(n):
        yn = y_tr.copy()
        idx = r2.choice(len(yn), int(rate*len(yn)), replace=False)
        yn[idx] = 1 - yn[idx]
        m_ = CatBoostClassifier(**HP).fit(Xtr_s, yn)
        s = m_.predict_proba(Xte_s)[:, 1]
        v.append((roc_auc_score(y_te, s), average_precision_score(y_te, s)))
    return np.array(v)

def rob_corruption(rate=0.20, n=CFG["n_robustness_runs"]):
    r2 = np.random.default_rng(SEED); v = []
    for _ in range(n):
        Xc = Xte_s.copy(); mask = r2.random(Xc.shape) < rate
        for j, c in enumerate(Xc.columns):
            k_ = int(mask[:, j].sum())
            if k_: Xc.loc[mask[:, j], c] = r2.choice(Xtr_s[c].values, k_, replace=True)
        s = model.predict_proba(Xc)[:, 1]
        v.append((roc_auc_score(y_te, s), average_precision_score(y_te, s)))
    return np.array(v)

ln, co = rob_label_noise(), rob_corruption()
iqr = lambda a: float(np.percentile(a, 75) - np.percentile(a, 25))
ROB = pd.DataFrame([
    {"Test": "Label noise, 10%",
     "Definition": "10% of training labels inverted uniformly at random",
     "AUC-ROC (median)": float(np.median(ln[:,0])), "IQR": iqr(ln[:,0]),
     "AU-PR (median)": float(np.median(ln[:,1])), "IQR ": iqr(ln[:,1]),
     "Runs": len(ln)},
    {"Test": "Feature corruption, 20%",
     "Definition": "20% of test values replaced by draws from the training marginal",
     "AUC-ROC (median)": float(np.median(co[:,0])), "IQR": iqr(co[:,0]),
     "AU-PR (median)": float(np.median(co[:,1])), "IQR ": iqr(co[:,1]),
     "Runs": len(co)}])
T10 = to_md(ROB, "table10_robustness",
    f"Table 10. Robustness. Reference without perturbation: AUC-ROC "
    f"{M['auc_roc']:.4f}, AU-PR {M['au_pr']:.4f}.")
print(ROB.round(4).to_string(index=False))
R["robustness"] = ROB.to_dict("records")

cross = {}
if df.ecosystem.nunique() > 1:
    for a, b in [("npm", "pypi"), ("pypi", "npm")]:
        A_, B_ = df[df.ecosystem == a], df[df.ecosystem == b]
        if not (len(A_) and len(B_) and A_.label.nunique() > 1 and B_.label.nunique() > 1):
            continue
        p_ = make_prep(SEL)
        m_ = CatBoostClassifier(**HP).fit(p_(A_)[SEL], A_.label.values)
        s = m_.predict_proba(p_(B_)[SEL])[:, 1]
        cross[f"{a}->{b}"] = {"AUC-ROC": roc_auc_score(B_.label.values, s),
                              "AU-PR": average_precision_score(B_.label.values, s),
                              "AU-PR baseline": float(B_.label.mean()),
                              "n test": len(B_), "n malicious": int(B_.label.sum())}
    auc_eco = roc_auc_score(df.label.values,
                            df.ecosystem.map(df.groupby("ecosystem")["label"].mean()).values)
    share = (auc_eco - .5)/(M["auc_roc"] - .5)
    T11 = to_md(pd.DataFrame([{"Direction": k, **v} for k, v in cross.items()] +
        [{"Direction": "Ecosystem of origin only (no metadata)", "AUC-ROC": auc_eco,
          "AU-PR": np.nan, "AU-PR baseline": np.nan, "n test": len(df),
          "n malicious": int(df.label.sum())}]), "table11_cross_ecosystem",
        f"Table 11. Cross-ecosystem transfer. A classifier using only the ecosystem "
        f"of origin attains AUC-ROC {auc_eco:.4f}, so {share:.1%} of discrimination "
        f"above chance is attributable to registry identity.")
    print(f"[b] AUC ecosysteme seul {auc_eco:.4f} -> part {share:.1%}")
    R["ecosystem_confound"] = {"auc_ecosystem_only": float(auc_eco),
                               "share": float(share), "cross": cross}

# =============================================================================
# 8. FIGURES
# =============================================================================
def save(fig, name):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight",
                    pad_inches=0.08, facecolor="white")
    plt.close(fig)

# Figure 2 : categories de features
cnt = pd.Series([category_of(f) for f in SEL]).value_counts()
imp = pd.Series({c: SHAPV[[f for f in SEL if category_of(f) == c]].sum()
                 for c in cnt.index})
order = imp.sort_values(ascending=False).index
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.4, 4.0),
                             gridspec_kw={"wspace": 0.42})
yv = np.arange(len(order))[::-1]
a1.barh(yv, [cnt[c] for c in order], color=BLUE, height=.62, edgecolor="white")
for yy, c in zip(yv, order):
    a1.text(cnt[c]+.1, yy, str(cnt[c]), va="center", fontsize=9, fontweight="bold")
a1.set_yticks(yv); a1.set_yticklabels(order, fontsize=8.6)
a1.set_xlabel("Number of selected features", fontsize=8.8)
a1.set_xlim(0, max(cnt)+1.4)
a1.set_title(f"(a)  Feature count per category (total = {K})",
             fontsize=9.2, fontweight="bold")
a2.barh(yv, [imp[c] for c in order], color=ORANGE, height=.62, edgecolor="white")
for yy, c in zip(yv, order):
    a2.text(imp[c]+.05, yy, f"{imp[c]:.2f}", va="center", fontsize=9, fontweight="bold")
a2.set_yticks(yv); a2.set_yticklabels([])
a2.set_xlabel(r"Aggregate mean $|$SHAP$|$", fontsize=8.8)
a2.set_xlim(0, imp.max()*1.22)
a2.set_title("(b)  Aggregate attribution per category", fontsize=9.2, fontweight="bold")
for ax in (a1, a2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=8.2)
save(fig, "figure2_feature_categories")

# Figure 3 : matrice de confusion
cm = confusion_matrix(y_te, (s_te >= thr).astype(int), labels=[0, 1])
assert cm.sum() == len(y_te)
fig, ax = plt.subplots(figsize=(5.4, 4.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Benign", "Malicious"], yticklabels=["Benign", "Malicious"],
            annot_kws={"fontsize": 13})
ax.set_xlabel("Predicted", fontsize=9.5); ax.set_ylabel("True label", fontsize=9.5)
ax.set_title(f"CatBoost, corrected protocol, threshold {thr:.2f} (n = {cm.sum()})",
             fontsize=9.6, fontweight="bold")
save(fig, "figure3_confusion_matrix")

# Figure 4 : balayage de k, deux panneaux avec bandes propres
ci_auc_w = CI["auc_roc"][1] - CI["auc_roc"][0]
fig, axes = plt.subplots(2, 1, figsize=(6.6, 5.8), sharex=True,
                         gridspec_kw={"hspace": .16})
for ax, col, lab, colr, mk, ls, w, tag in [
        (axes[0], "AU-PR", "AU-PR", BLUE, "o", "-", CI_W, "(a)"),
        (axes[1], "AUC-ROC", "AUC-ROC", ORANGE, "s", "--", ci_auc_w, "(b)")]:
    r_ = float(SWEEP.loc[SWEEP.k == K, col].iloc[0])
    ax.axhspan(r_-w, r_+w, color=colr, alpha=.11, zorder=0,
               label=f"within 95% CI at $k$={K} (±{w:.4f})")
    ax.plot(SWEEP.k, SWEEP[col], marker=mk, ls=ls, color=colr, lw=1.8, ms=6,
            label=lab, zorder=3)
    ax.axvline(K, color=GREY, ls=":", lw=1.3, zorder=2)
    lo_, hi_ = min(SWEEP[col].min(), r_-w)-.002, max(SWEEP[col].max(), r_+w)+.002
    ax.set_ylim(lo_, hi_); ax.set_ylabel(lab, fontsize=9.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=.26); ax.set_axisbelow(True); ax.tick_params(labelsize=8.3)
    ax.legend(fontsize=7.5, loc="lower right")
    ax.text(.008, .92, tag, transform=ax.transAxes, fontsize=9.4, fontweight="bold")
axes[1].set_xticks(list(SWEEP.k))
axes[1].set_xticklabels([str(int(x)) for x in SWEEP.k], fontsize=8.6)
axes[1].set_xlabel("Number of selected features ($k$)", fontsize=9.4)
axes[0].set_title("Performance versus feature-set size (test partition)",
                  fontsize=10.2, fontweight="bold", pad=8)
fig.text(.5, .03, f"Ecosystem-identifying attributes removed before selection. "
         f"AU-PR baseline = {M['au_pr_baseline']:.3f}, below both ranges.",
         ha="center", va="top", fontsize=7.4, style="italic", color="#444")
save(fig, "figure4_k_sweep")

# Figure 5 : SHAP summary
fig = plt.figure(figsize=(7.2, 5.6))
shap.summary_plot(sv, Xte_s, show=False, max_display=K)
plt.title(f"SHAP summary, CatBoost, corrected protocol (n = {M['n']})",
          fontsize=10, fontweight="bold")
plt.tight_layout(); save(plt.gcf(), "figure5_shap_summary")

# Figure 6 : diagramme de fiabilite
def ece(y, s, bins=10):
    e_, n_ = 0., len(y); edges = np.linspace(0, 1, bins+1)
    for i in range(bins):
        m_ = (s > edges[i]) & (s <= edges[i+1])
        if m_.sum(): e_ += (m_.sum()/n_)*abs(y[m_].mean() - s[m_].mean())
    return float(e_)
frac, meanp = calibration_curve(y_te, s_te, n_bins=10, strategy="quantile")
ECE = ece(y_te, s_te)
fig, ax = plt.subplots(figsize=(5.4, 4.8))
ax.plot(meanp, frac, "o-", color=BLUE, label="CatBoost")
ax.plot([0, 1], [0, 1], "k--", alpha=.5, lw=.9, label="perfectly calibrated")
ax.set_xlabel("Mean predicted probability", fontsize=9.4)
ax.set_ylabel("Observed fraction of positives", fontsize=9.4)
ax.set_title(f"Reliability diagram — Brier {M['brier']:.4f}, ECE {ECE:.4f}",
             fontsize=9.8, fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=.3); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "figure6_calibration")
R["calibration"] = {"brier": M["brier"], "ece_10bins": ECE,
                    "curve": [{"mean_predicted": float(a), "observed": float(b)}
                              for a, b in zip(meanp, frac)]}

# SHAP : stabilite et accord avec la permutation
RK = []
for _ in range(CFG["n_stability_runs"]):
    keep = set(rng.choice(uq, size=len(uq), replace=True))
    mk = np.array([g in keep for g in train.group_id.values])
    if len(np.unique(y_tr[mk])) < 2: continue
    m_ = CatBoostClassifier(**HP).fit(Xtr_s[mk], y_tr[mk])
    v_ = shap.TreeExplainer(m_).shap_values(Xte_s); v_ = v_[1] if isinstance(v_, list) else v_
    RK.append(pd.Series(np.abs(v_).mean(0), index=SEL).rank(ascending=False))
RKdf = pd.concat(RK, axis=1)
stab = float(np.mean([spearmanr(RKdf.iloc[:, i], RKdf.iloc[:, j]).correlation
                      for i, j in itertools.combinations(range(RKdf.shape[1]), 2)]))
permi = permutation_importance(model, Xte_s, y_te, n_repeats=10, random_state=SEED,
                               scoring="average_precision")
agree = float(spearmanr(SHAPV.loc[SEL], permi.importances_mean).correlation)
print(f"\nSHAP : stabilite des rangs {stab:.3f} | accord permutation {agree:.3f}")
R["shap"] = {"global": SHAPV.round(4).to_dict(), "rank_stability_spearman": stab,
             "agreement_permutation_spearman": agree}

# =============================================================================
# 9. EXPORT
# =============================================================================
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, list): return [clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, (np.bool_, bool)): return bool(o)
    if isinstance(o, np.str_): return str(o)
    return o

R["generated_at"] = datetime.now(timezone.utc).isoformat()
json.dump(clean(R), open(f"{OUT}/final_results.json", "w"), indent=2, default=str)
pd.concat([p[["package", "version", "ecosystem", "group_id"]].assign(split=n)
           for p, n in [(train, "train"), (val, "val"), (test, "test")]]
          ).to_csv(f"{OUT}/final_partitions.csv", index=False)
open(f"{TAB}/ALL_TABLES.md", "w").write("\n\n---\n\n".join(MD[k] for k in sorted(MD)))

guide = f"""# Valeurs a reporter — jcp-4466772

Genere le {R['generated_at']}. Toute valeur du manuscrit doit venir d'ici.

## Tables (dans {TAB}/, aussi en Markdown dans ALL_TABLES.md)

| Fichier | Section du manuscrit |
|---|---|
| table01_dataset, table01b_partitions | 2.3, Table 1 |
| table02_reduction | 2.4, Table 2 |
| table03_features | 3.1, Table 3 |
| table04_performance | 3.2, Table 4 |
| table05_prevalence | 3.6, Table 5 |
| table06_gain_decomposition | 3.2, Table 6 |
| table07_k_sweep | 3.2, Table 7 |
| table08_paired_tests | 3.2, Table 8 |
| table09_family_ablation | 3.3, Table 9 |
| table10_robustness | 3.5, Table 10 |
| table11_cross_ecosystem | 3.5, Table 11 |

## Figures (dans {FIG}/, en PNG 600 dpi, PDF et SVG)

| Fichier | Section |
|---|---|
| figure2_feature_categories | 3.1 |
| figure3_confusion_matrix | 3.2 |
| figure4_k_sweep | 3.2 |
| figure5_shap_summary | 3.3 |
| figure6_calibration | 3.6 |

La Figure 1 (schema du pipeline) se genere separement.

## Chiffres pour l'abstract

- Espace de features : {len(F_FULL)} -> {len(F_POST)} -> {K}
- AU-PR {M['au_pr']:.4f} (baseline {M['au_pr_baseline']:.4f}, lift {M['au_pr_lift']:.4f})
- AUC-ROC {M['auc_roc']:.4f} | MCC {M['mcc']:.4f}
- Rappel au seuil {thr:.3f} : {mt['recall']:.3f} | precision {mt['precision']:.3f}
- VPP a {pi:.1%} : {op['PPV']:.4f} ({op['alerts_per_10k']:.0f} alertes/10k)
- Pre-filtre {g_full-g_post:+.4f} | selection {g_post:+.4f} | IC {CI_W:.4f}
- Selection {'au-dessus du' if sel_sig else 'sous le'} bruit
- Stabilite Jaccard {R['stability']['mean_jaccard']:.3f}, {R['stability']['n_stable_80pct']}/{K} stables
- Confondant ecosysteme : {R.get('ecosystem_confound', {}).get('share', float('nan')):.1%}
- Calibration : Brier {M['brier']:.4f}, ECE {ECE:.4f}
- SHAP : stabilite {stab:.3f}, accord permutation {agree:.3f}

## Phrases dont la formulation depend du resultat

Selection : {'"the selection contributes measurably beyond the pre-filter"' if sel_sig
             else '"the selection does not exceed the pre-filter beyond noise"'}
k : {'"k = 18 and k = 40 are equivalent; smaller budgets are significantly worse"'
     if abs(SWEEP.iloc[0]['Delta vs k=18']) > CI_W
     else '"performance saturates below k = 18"'}
"""
open(f"{OUT}/MANUSCRIPT_VALUES.md", "w").write(guide)

print(f"""
======================================================================
GENERATION TERMINEE
----------------------------------------------------------------------
Attributs   {len(F_FULL)} -> {len(F_POST)} -> {K}
AU-PR       {M['au_pr']:.4f}  baseline {M['au_pr_baseline']:.4f}  lift {M['au_pr_lift']:.4f}
AUC-ROC     {M['auc_roc']:.4f}   MCC {M['mcc']:.4f}
Test        {M['n']} ({M['TP']+M['FN']} mal / {M['TN']+M['FP']} ben)
Seuil       {thr:.3f} | rappel {mt['recall']:.3f} | ecart {gap:+.3f}
VPP a 0.1%  {op['PPV']:.4f} ({op['alerts_per_10k']:.0f} alertes/10k)
Pre-filtre  {g_full-g_post:+.4f} | selection {g_post:+.4f} | IC {CI_W:.4f}
            -> la selection est {'AU-DESSUS du' if sel_sig else 'SOUS le'} bruit
----------------------------------------------------------------------
11 tables   {TAB}/ (CSV + ALL_TABLES.md pret a coller)
5 figures   {FIG}/ (PNG 600 dpi, PDF, SVG)
Guide       {OUT}/MANUSCRIPT_VALUES.md
Tracabilite {OUT}/final_results.json
======================================================================""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
3330 versions apres appariement temporel | {'npm': ['2024-03-22', '2026-08-05'], 'pypi': ['2022-01-10', '2026-07-19']}
1490 groupes | train 1713 val 649 test 968 (691 mal / 277 ben)
Ecosystem  Versions  Malicious  Benign Malicious %
      NPM      2505       2092     413       83.5%
     PYPI       825         79     746        9.6%
    Total      3330       2171    1159       65.2%

espace complet 456 | proxies 404 | post-filtre 52
                               Stage Features in  Features out Removed
  Extracted point-in-time attributes           -           456       -
   Ecosystem pre-filter (vman scope)         456            52     404
Embedded selection, top-k (a priori)          52            18      34

AU-PR 0.9919 CI [0.9886,0.9945] | AUC 0.9766 | MCC 0.8317

Table 3 : 18 features, dominante = pit_has_repository (1.149)

!! ecart de transfert +0.39

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00
